# Chapter 4 — Similarity Is a Decision

**Book alignment:** Embeddings From First Principles, Chapter 4

**Question this notebook isolates:** For two candidate matches, does the *metric* decide the
winner (Euclidean can pick a different document than cosine) — and does that freedom vanish
once the encoder L2-normalises its output, as the committed RELATE metric sweep shows
(spread 0.0013 nDCG@10 across eight conditions)?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave: str, name: str) -> dict:
    return json.loads((EXP / wave / "artifacts" / name).read_text())

## 1. Two vectors, four answers

In [ ]:
x = np.array([2.0, 0.0, 0.0])     # short doc, one strong topic
y = np.array([6.0, 0.1, 0.0])     # long doc, same topic, more of it
z = np.array([0.0, 2.0, 0.0])     # short doc, different topic

def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"                x-y      x-z")
print(f"dot product   {x @ y:6.2f}   {x @ z:6.2f}   -> y")
print(f"euclidean     {np.linalg.norm(x - y):6.2f}   {np.linalg.norm(x - z):6.2f}   -> z is CLOSER")
print(f"cosine        {cos(x, y):6.2f}   {cos(x, z):6.2f}   -> y")

assert np.linalg.norm(x - z) < np.linalg.norm(x - y)      # euclidean disagrees with cosine
assert cos(x, y) > cos(x, z)
print("\nnothing about the documents changed - the decision rule changed")

## 2. The algebra: cosine == normalized dot == monotone in normalized L2

In [ ]:
def unit(v): return v / np.linalg.norm(v)
xh, yh = unit(x), unit(y)
assert np.isclose(cos(x, y), xh @ yh)
assert np.isclose(np.linalg.norm(xh - yh) ** 2, 2 - 2 * cos(x, y))   # ||x^-y^||^2 = 2 - 2 cos
print("normalize -> cosine, dot, and euclidean all give the same ranking")

## 3. On a normalized encoder, the metric choice is close to inert (RELATE, Wave 1)

In [ ]:
ms = art("wave1", "metric-sweep.json")
for cond, v in ms["conditions"].items():
    print(f"  {cond:16} nDCG@10 all={v['ndcg10_all']:.4f}  hardneg={v['ndcg10_hardneg']:.4f}")

vals = [v["ndcg10_all"] for v in ms["conditions"].values()]
spread = max(vals) - min(vals)
print(f"\nspread across all 8 conditions: {spread:.4f}")
assert spread < 0.002
# 'raw' and 'normalized' are identical numbers because the sentence-transformer already L2-normalizes
assert ms["conditions"]["cosine_raw"] == ms["conditions"]["cosine_norm"]
print("the decision that matters is whether to normalize AT ALL - and the model made it for you")

## What we earned

"Similar" is the output of a metric applied to a representation — choose both deliberately.
Euclidean and cosine genuinely rank differently on magnitude-carrying vectors. But a modern
encoder L2-normalises its output, which collapses cosine, dot, and Euclidean to one ranking:
on RELATE the eight metric conditions span 0.0013 nDCG@10. The length-bias story is real
only for embeddings that keep their magnitude.

**Notebook 05 / Chapter 5** goes inside a single vector and asks what dimension 173 means —
answer: usually nothing that survives a rotation.